# Notebook 2 — Logistic Regression

**50 minutes.** Each cell tells you what the output should look like,
so you can check yourself without waiting for a hand.

Dataset: **diamonds** — 53,940 stones. We predict whether a diamond sells for
**more than $5,000**, from its size and clarity.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.facecolor": "#08080A", "axes.facecolor": "#08080A",
    "savefig.facecolor": "#08080A", "text.color": "#F5F5F7",
    "axes.labelcolor": "#F5F5F7", "axes.edgecolor": "#5A5A66",
    "xtick.color": "#8A8A99", "ytick.color": "#8A8A99",
    "axes.grid": True, "grid.color": "#1C1C22", "figure.figsize": (7, 4),
})
RED, BLUE, GREEN, YELLOW, WHITE = "#FF3B3B", "#3B9EFF", "#35E07E", "#FFD23F", "#F5F5F7"

---
## Load and build the target

1. Load `sns.load_dataset("diamonds")` into `dia`.
2. Create a column `premium` that is `1` when `price` is above 5000, else `0`.
   *Hint: `(dia["price"] > 5000).astype(int)`*
3. Clarity is text (`SI2`, `VS1`, ...). Turn it into a number 1-8 using the map provided.

**Watch out:** seaborn stores `clarity` as a pandas *Categorical*, and `.map()` on a Categorical
hands you back another Categorical instead of plain numbers — which quietly breaks every
calculation later. Chain `.astype(int)` on the end to force real integers.

In [ ]:
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]
clarity_map = {c: i + 1 for i, c in enumerate(clarity_order)}


dia = sns.load_dataset("diamonds")
dia["premium"] = (dia["price"] > 5000).astype(int)
dia["clarity_rank"] = pd.Series(list(clarity_map.values()))

print(dia.shape)                              # expected: (53940, 12)
print(f"premium rate: {dia.premium.mean():.3f}")   # expected: 0.273

## The baseline you must beat

Before modelling anything: if you ignored the data entirely and always guessed the
**more common** class, what accuracy would you get?

Compute it. Write it on a sticky note. This is the number your model has to beat.

In [ ]:
baseline = ((dia['premium'] == 0).sum()/((dia['premium']==0).sum() + (dia['premium']==1).sum()))
print(f"always guessing 'not premium' gets {baseline:.1%}")   # expected: 72.7%

## Split, then scale

Two features: `carat` and `clarity_rank`.

1. `train_test_split` with `test_size=0.25`, `random_state=42`, `stratify=y`
2. Fit a `StandardScaler` **on the training set only**, then transform both.

Fitting the scaler on all the data leaks test information into training. It is a
silent, common, marks-losing bug.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = dia[["carat", "clarity_rank"]].values
y = dia["premium"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(X_train_s.shape, X_test_s.shape)          # expected: (40455, 2) (13485, 2)
print(f"train mean ~ {X_train_s.mean():.2f}")   # expected: about 0.00

---
## Write `sigmoid`

```
sigmoid(z) = 1 / (1 + e^(-z))
```

In [ ]:
import math
def sigmoid(z):
    return 1/(1 + math.e**(z))

print(sigmoid(0))                    # expected: 0.5
print(np.round(sigmoid(np.array([-6, 0, 6])), 3))   # expected: [0.002 0.5 0.998]

Run this to see the squash. **Nothing to fill in.**

In [ ]:
z = np.linspace(-8, 8, 200)
plt.plot(z, sigmoid(z), color=GREEN, lw=2.5)
plt.axhline(0.5, color=YELLOW, ls="--", lw=1)
plt.xlabel("z = w·x + b"); plt.ylabel("probability")
plt.show()

## Write `log_loss`

For each sample: if the true answer is 1, the penalty is `-log(probability)`.
If the true answer is 0, the penalty is `-log(1 - probability)`. Return the mean.

*Hint: `np.where(y_true == 1, -np.log(p), -np.log(1 - p))`*

In [ ]:
def log_loss(y_true, prob):
    prob = np.clip(prob, 1e-7, 1 - 1e-7)
    return (y_true * np.log(prob) + (1 - y_true) * np.log(1 - prob)).mean()
        

test_prob = np.array([0.9, 0.1, 0.5])
test_true = np.array([1,   0,   1])
print(f"{log_loss(test_true, test_prob):.3f}")   # expected: 0.301

---
## The gradient descent loop

Same shape as notebook 1, with two differences: predictions go through `sigmoid`,
and the gradient formulas change slightly (no factor of `-2` this time).


 `error = prob - y`
 `dw = (X.T @ error) / n_samples`
 `db = mean(error)`
 step against them: `w = w - alpha * dw`, same for `b`

Fill in the four marked lines.

In [ ]:

def train_logistic(X, y, alpha=1.0, steps=3000):
    
    n_samples, n_features = X.shape
    w = np.zeros(n_features)
    b = 0.0
    history = []
    for step in range(steps):
        z = np.matmul(X, w) + b                  # linear part
        prob = sigmoid(z)               # squash into 0-1
        error = prob-y               # prob - truth
        dw = (X.T @ error) / n_samples                # gradient for w
        db = np.mean(error)         # gradient for b
        w = w - alpha * dw
        b = b - alpha * db
        history.append(log_loss(y, prob))
    return w, b, history

w, b, history = train_logistic(X_train_s, y_train)
print(f"w = {np.round(w, 3)}   b = {b:.3f}")   # expected: w = [8.172 2.077]  b = -3.535
print(f"final loss = {history[-1]:.4f}")        # expected: roughly 0.1209

### Why alpha = 1.0 and 3000 steps here?

Same story as notebook 1. Moving `w` and `b` together makes the error surface a narrow
valley rather than a clean bowl, so it needs a bigger step and more iterations than the
single-variable slide animation suggested. Try `alpha=0.1, steps=1000` and you will land
at `w ≈ [4.2, 1.0]` — a real answer, but a noticeably worse one, because it stopped short
of the bottom rather than diverging. Slower convergence is a gentler failure than the
exploding blow-up you saw in notebook 1's TODO 7, but it is still a bug worth recognising.

## Plot the loss curve

Plot `history`. x-axis is the step number, y-axis is log loss.

In [ ]:
plt.plot(list(range(3000)), history)
plt.xlabel("step"); plt.ylabel("log loss")
plt.show()
# It should drop fast, then flatten. That flattening is convergence.

---
## Predict on the test set

Compute `prob_test`, the predicted probability for every test diamond.
Then turn it into 0/1 predictions at threshold 0.5 and compute accuracy.

In [ ]:
z_test = np.matmul(X_test_s, w) + b
prob_test = sigmoid(z_test)
preds_test = np.array([0 if i < 0.5 else 1 for i in prob_test])             # 1 where prob_test >= 0.5, else 0
accuracy = ((y_test == preds_test).sum()/(len(y_test)))

print(f"accuracy {accuracy:.4f}")            # expected: about 0.9489
print(f"baseline {baseline:.4f}")            # 0.7273 -- did you beat it?



## Confusion matrix

Use `confusion_matrix(y_test, preds_test)` and unpack the four numbers.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, preds_test)
tn, fp, fn, tp = cm.ravel()

print(f"true negatives  {tn}")     # expected: 9491
print(f"false positives {fp}")     # expected: 316
print(f"false negatives {fn}")     # expected: 373
print(f"true positives  {tp}")     # expected: 3305

## Precision and recall, by hand

From those four numbers:

```
precision = tp / (tp + fp)      of the ones you flagged, how many were right
recall    = tp / (tp + fn)      of the ones that mattered, how many you caught
```

In [ ]:
precision = tp/(tp+fn)
recall = tp/(tp+fn)
print(f"precision {precision:.3f}   recall {recall:.3f}")   # expected: 0.913, 0.899



## Move the threshold

Write a function that takes a threshold and returns precision and recall at that cutoff.
Then loop over a range of thresholds and plot both curves.

In [ ]:
def at_threshold(t):
    preds = np.array([0 if i < t else 1 for i in prob_test])                  # 1 where prob_test >= t, else 0
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    return tp / (tp + fp), tp / (tp + fn)

ts = np.linspace(0.05, 0.95, 40)
precisions, recalls = zip(*[at_threshold(t) for t in ts])

plt.plot(ts, precisions, color=GREEN, lw=2, label="precision")
plt.plot(ts, recalls, color=BLUE, lw=2, label="recall")
plt.xlabel("threshold"); plt.legend(); plt.show()

## Pick one, and justify it

You run a jewellery shop. Flagging a stone as "premium" means you price it high.

- A **false positive** means an overpriced stone that sits in the case for months.
- A **false negative** means you sold a good stone too cheap.

Choose a threshold and defend it in one sentence.

**I would lower the threshold to somewhat near 0.45:**

*"I'd lower the threshold below 0.5, because a false negative permanently loses money on a genuinely valuable stone, while a false positive only costs shelf time on inventory that can still be sold later*

---
## Now the three-line version

Everything you just wrote, scikit-learn does in three lines. Fit a `LogisticRegression`
on the same scaled training data and compare its accuracy to yours.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()
clf.fit(X_train_s, y_train)

acc_sklearn = clf.score(X_test_s, y_test)
print(f"sklearn:  {acc_sklearn:.4f}")
print(f"yours:    {accuracy:.4f}")
# These should be very close. You just reimplemented a library.

## Read the coefficients

Print `clf.coef_`, then exponentiate them (`np.exp`) to get **odds multipliers**.

In [ ]:
print("coefficients:", clf.coef_)        # expected: [8.077 2.052]
print("odds multipliers:", np.exp(clf.coef_))      # expected: [3220.34 7.79]
# Which feature matters more -- size or clarity?

---
## Stretch goal — draw the decision boundary

Reproduce slide 27. The boundary is where `w1*x1 + w2*x2 + b = 0`, so solve for `x2`:

```
x2 = -(b + w1*x1) / w2
```

Remember your `clf` was fitted on **scaled** data, so compute the line in scaled space
and convert back with `scaler.mean_` and `scaler.scale_`.

In [ ]:
# stretch